In [3]:
#read json file from LH

from pyspark.sql import functions as F
df = spark.read.option("multiline", "true").json("Files/pokemon.json")

display(df)
#nested columns found:abilities,cries,forms,game_indices,moves,past_abilities,past_stats,species,stats,types

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ff667278-d9a4-4f7b-aae2-57b96f1a5462)

In [4]:
df.printSchema()


StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 6, Finished, Available, Finished, False)

root
 |-- abilities: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- ability: struct (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- url: string (nullable = true)
 |    |    |-- is_hidden: boolean (nullable = true)
 |    |    |-- slot: long (nullable = true)
 |-- base_experience: long (nullable = true)
 |-- cries: struct (nullable = true)
 |    |-- latest: string (nullable = true)
 |    |-- legacy: string (nullable = true)
 |-- forms: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- url: string (nullable = true)
 |-- game_indices: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- game_index: long (nullable = true)
 |    |    |-- version: struct (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- url: string (nullable = true)
 |-- height: long (nullable

In [6]:
df = spark.read.option("multiline","true").json("Files/pokemon.json")
df.createOrReplaceTempView("pokemon")
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 8, Finished, Available, Finished, False)

+----------------------------------+--------------------+-----------+
|namespace                         |tableName           |isTemporary|
+----------------------------------+--------------------+-----------+
|`MS Fabric w space`.lh_pokemon.dbo|pokemon             |false      |
|`MS Fabric w space`.lh_pokemon.dbo|pokemon_abilities   |false      |
|`MS Fabric w space`.lh_pokemon.dbo|pokemon_cries       |false      |
|`MS Fabric w space`.lh_pokemon.dbo|pokemon_forms       |false      |
|`MS Fabric w space`.lh_pokemon.dbo|pokemon_game_indices|false      |
|`MS Fabric w space`.lh_pokemon.dbo|pokemon_main        |false      |
|`MS Fabric w space`.lh_pokemon.dbo|pokemon_moves       |false      |
|`MS Fabric w space`.lh_pokemon.dbo|pokemon_past_stats  |false      |
|`MS Fabric w space`.lh_pokemon.dbo|pokemon_species     |false      |
|`MS Fabric w space`.lh_pokemon.dbo|pokemon_sprites     |false      |
|`MS Fabric w space`.lh_pokemon.dbo|pokemon_stats       |false      |
|`MS Fabric w space`

In [12]:
CREATE OR REPLACE table pokemon_abilities AS
SELECT
    p.id,
    p.name,
    a.ability.name AS ability_name,
    a.is_hidden,
    a.slot
FROM pokemon p
LATERAL VIEW EXPLODE(p.abilities) x AS a

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
SHOW TABLES;

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 10, Finished, Available, Finished, False)

<Spark SQL result set with 14 rows and 3 fields>

In [10]:
df.write.mode("overwrite").saveAsTable("pokemon")

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 12, Finished, Available, Finished, False)

In [13]:
CREATE or replace TABLE pokemon_cries AS
SELECT
    id,
    cries.latest AS cry_latest,
    cries.legacy AS cry_legacy
FROM pokemon


StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [19]:
CREATE OR REPLACE TABLE pokemon_forms AS
SELECT
    p.id,
    f.name AS form_name,
    f.url AS form_url
FROM pokemon p
LATERAL VIEW EXPLODE(p.forms) x AS f;

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [20]:
CREATE OR REPLACE TABLE pokemon_game_indices AS
SELECT
    p.id,
    gi.game_index,
    gi.version.name AS version_name,
    gi.version.url AS version_url
FROM pokemon p
LATERAL VIEW EXPLODE(p.game_indices) x AS gi;

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 22, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [21]:
CREATE OR REPLACE TABLE pokemon_moves AS
SELECT
    p.id,
    p.name,

    m.move.name AS move_name,
    m.move.url AS move_url,

    v.level_learned_at,

    v.move_learn_method.name AS learn_method_name,
    v.move_learn_method.url AS learn_method_url,

    v.version_group.name AS version_group_name,
    v.version_group.url AS version_group_url

FROM pokemon p
LATERAL VIEW EXPLODE(p.moves) mv AS m
LATERAL VIEW EXPLODE(m.version_group_details) vd AS v;


StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 23, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [22]:
CREATE OR REPLACE TABLE pokemon_past_stats AS
SELECT
    p.id,
    p.name,

    ps.generation.name AS generation_name,
    ps.generation.url AS generation_url,

    s.stat.name AS stat_name,
    s.stat.url AS stat_url,

    s.base_stat,
    s.effort

FROM pokemon p
LATERAL VIEW EXPLODE(p.past_stats) pst AS ps
LATERAL VIEW EXPLODE(ps.stats) st AS s;

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 24, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [15]:
CREATE or replace TABLE pokemon_species AS
SELECT
    id,
    species.name AS species_name,
    species.url AS species_url
FROM pokemon

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [16]:
CREATE or replace  TABLE pokemon_sprites AS
SELECT
    id,
    sprites.front_default,
    sprites.front_shiny,
    sprites.back_default,
    sprites.back_shiny
FROM pokemon

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 18, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [17]:
CREATE or replace  TABLE pokemon_stats AS
SELECT
    p.id,
    s.stat.name AS stat_name,
    s.stat.url AS stat_url,
    s.base_stat,
    s.effort
FROM pokemon p
LATERAL VIEW EXPLODE(p.stats) st AS s

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [18]:
CREATE or replace  TABLE pokemon_types AS
SELECT
    p.id,
    t.slot,
    t.type.name AS type_name,
    t.type.url AS type_url
FROM pokemon p
LATERAL VIEW EXPLODE(p.types) tp AS t

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 20, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [23]:
SELECT
    p.id,
    p.name,

    pa.ability_name,

    pm.move_name,

    pt.type_name,

    ps.stat_name,
    ps.base_stat,

    pf.form_name,

    pg.game_index,

    pc.cry_latest,
    pc.cry_legacy

FROM pokemon p

LEFT JOIN pokemon_abilities pa
    ON p.id = pa.id

LEFT JOIN pokemon_moves pm
    ON p.id = pm.id

LEFT JOIN pokemon_types pt
    ON p.id = pt.id

LEFT JOIN pokemon_stats ps
    ON p.id = ps.id

LEFT JOIN pokemon_forms pf
    ON p.id = pf.id

LEFT JOIN pokemon_game_indices pg
    ON p.id = pg.id

LEFT JOIN pokemon_cries pc
    ON p.id = pc.id

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 11 fields>

In [24]:
--Pokémon-Level KPIs
SELECT
    COUNT(*) AS pokemon_count,
    AVG(weight) AS avg_weight,
    AVG(height) AS avg_height
FROM pokemon

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 26, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [25]:
--Move KPIs
SELECT
    id,
    COUNT(*) AS move_count
FROM pokemon_moves
GROUP BY id
ORDER BY move_count DESC

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 27, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [ ]:
--Ability KPIs----
SELECT
    ability_name,
    COUNT(*) AS pokemon_count
FROM pokemon_abilities
GROUP BY ability_name
ORDER BY pokemon_count DESC

In [26]:
--Type KPIs
SELECT
    type_name,
    COUNT(DISTINCT id) AS pokemon_count
FROM pokemon_types
GROUP BY type_name


StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 28, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [29]:
--Stat KPIs
SELECT
    id,
    SUM(base_stat) AS total_stats
FROM pokemon_stats
GROUP BY id
ORDER BY total_stats DESC

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 31, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [27]:
--Generation KPIs
SELECT
    generation_name,
    AVG(base_stat) AS avg_stat
FROM pokemon_past_stats
GROUP BY generation_name

StatementMeta(, 7c3e27e8-5c57-40b5-802e-a2f811699f01, 29, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>